# PeptideScreen — OpenMM MD Stability (Colab)

Runs a 50ns MD simulation on a peptide-receptor complex from the pipeline.

**Before running:** Upload your candidate PDB and receptor PDB from the pipeline's run directory.

Files needed from your run:
- `{run_dir}/candidates/folded_structures/{CANDIDATE_ID}.pdb` (peptide)
- `{run_dir}/docking/receptor/{PDB_ID}_receptor.pdb` (receptor)

Or upload a pre-built complex PDB directly.

In [ ]:
# Cell 1: Install dependencies
!pip install -q openmm pdbfixer

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/MD_Results', exist_ok=True)

In [ ]:
# Cell 3: Keep Colab alive
%%javascript
function ClickConnect(){
  console.log("Keeping alive");
  document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)

In [ ]:
# Cell 4: Upload and build complex
# =====================================================
# OPTION A: Upload separate receptor + peptide PDBs
# Upload them to Colab, then set paths here:
#
# from google.colab import files
# uploaded = files.upload()  # upload both PDBs
#
# OPTION B: Upload a pre-built complex PDB directly
# =====================================================

# EDIT THESE for your candidate:
CANDIDATE_ID = "NRF2KEAP-001"   # from pipeline output
RECEPTOR_PDB = "receptor.pdb"    # uploaded receptor PDB
PEPTIDE_PDB = "peptide.pdb"      # uploaded peptide PDB
SIM_NS = 50                       # simulation length

OUTPUT_DIR = f"/content/drive/MyDrive/MD_Results/{CANDIDATE_ID}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Build complex by combining receptor + peptide
COMPLEX_PDB = f"/content/{CANDIDATE_ID}_complex.pdb"

receptor_lines = []
peptide_lines = []

with open(RECEPTOR_PDB) as f:
    for line in f:
        if line.startswith(('ATOM', 'HETATM')):
            receptor_lines.append(line)

with open(PEPTIDE_PDB) as f:
    for line in f:
        if line.startswith(('ATOM', 'HETATM')):
            peptide_lines.append(line)

with open(COMPLEX_PDB, 'w') as f:
    for line in receptor_lines:
        f.write(line)
    f.write('TER\n')
    for line in peptide_lines:
        f.write(line)
    f.write('TER\nEND\n')

print(f'Complex built: {COMPLEX_PDB}')
print(f'Receptor atoms: {len(receptor_lines)}')
print(f'Peptide atoms: {len(peptide_lines)}')

In [ ]:
# Cell 5: Run MD simulation
import signal
import sys
import time

from openmm import unit, LangevinMiddleIntegrator, MonteCarloBarostat
from openmm.app import (
    ForceField, Modeller, Simulation, PDBFile, PME, HBonds,
    DCDReporter, StateDataReporter, CheckpointReporter
)
from openmm import Platform
from pdbfixer import PDBFixer

# Paths
solvated_pdb = f"{OUTPUT_DIR}/{CANDIDATE_ID}_solvated.pdb"
traj_file = f"{OUTPUT_DIR}/{CANDIDATE_ID}_trajectory.dcd"
log_file = f"{OUTPUT_DIR}/{CANDIDATE_ID}_energy.csv"
checkpoint_file = f"{OUTPUT_DIR}/{CANDIDATE_ID}_checkpoint.chk"
final_pdb = f"{OUTPUT_DIR}/{CANDIDATE_ID}_final.pdb"

dt = 0.002  # ps
total_steps = int(SIM_NS * 1000 / dt)
report_interval = 25000     # 50 ps
traj_interval = 25000       # 50 ps
checkpoint_interval = 250000 # 500 ps

# Check for resume
resume = os.path.exists(solvated_pdb) and os.path.exists(checkpoint_file)

forcefield = ForceField('charmm36.xml', 'charmm36/water.xml')

if resume:
    print(f'[RESUME] Loading saved system...')
    pdb_loaded = PDBFile(solvated_pdb)
    topology = pdb_loaded.topology
    positions = pdb_loaded.positions
else:
    print('[1/6] Fixing structure...')
    fixer = PDBFixer(filename=COMPLEX_PDB)
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.4)
    fixer.removeHeterogens(keepWater=False)

    print('[2/6] Solvating...')
    modeller = Modeller(fixer.topology, fixer.positions)
    modeller.addSolvent(forcefield, model='tip3p',
                        padding=1.0*unit.nanometers,
                        ionicStrength=0.15*unit.molar)
    topology = modeller.topology
    positions = modeller.positions
    print(f'  Total atoms: {topology.getNumAtoms()}')

    with open(solvated_pdb, 'w') as f:
        PDBFile.writeFile(topology, positions, f)

print('[3/6] Creating system...')
system = forcefield.createSystem(topology, nonbondedMethod=PME,
                                  nonbondedCutoff=1.0*unit.nanometers,
                                  constraints=HBonds)
system.addForce(MonteCarloBarostat(1*unit.atmospheres, 310.15*unit.kelvin))

integrator = LangevinMiddleIntegrator(310.15*unit.kelvin,
                                       1.0/unit.picosecond,
                                       dt*unit.picoseconds)

# Select platform
for pname in ['CUDA', 'OpenCL', 'CPU']:
    try:
        platform = Platform.getPlatformByName(pname)
        print(f'  Platform: {pname}')
        break
    except: continue

simulation = Simulation(topology, system, integrator, platform)
simulation.context.setPositions(positions)

if resume:
    simulation.loadCheckpoint(checkpoint_file)
    current_step = simulation.context.getState().getStepCount()
    remaining = total_steps - current_step
    print(f'  Resumed at {current_step * dt / 1000:.1f} ns, {remaining} steps remaining')
    # New files for resumed segment
    traj_file = f"{OUTPUT_DIR}/{CANDIDATE_ID}_trajectory_resumed_{current_step}.dcd"
    log_file = f"{OUTPUT_DIR}/{CANDIDATE_ID}_energy_resumed_{current_step}.csv"
else:
    remaining = total_steps
    print('[4/6] Minimizing...')
    simulation.minimizeEnergy()

    print('[5/6] Equilibrating...')
    simulation.context.setVelocitiesToTemperature(310.15*unit.kelvin)
    simulation.step(50000)  # NVT 100ps
    simulation.step(50000)  # NPT 100ps

# Reporters
simulation.reporters.append(DCDReporter(traj_file, traj_interval))
simulation.reporters.append(StateDataReporter(
    log_file, report_interval, step=True, time=True,
    potentialEnergy=True, kineticEnergy=True, totalEnergy=True,
    temperature=True, volume=True, speed=True, separator=','))
simulation.reporters.append(StateDataReporter(
    sys.stdout, report_interval * 4, step=True, time=True,
    temperature=True, speed=True, remainingTime=True, totalSteps=total_steps))
simulation.reporters.append(CheckpointReporter(checkpoint_file, checkpoint_interval))

print(f'[6/6] Production MD ({SIM_NS} ns)...')
start = time.time()
simulation.step(remaining)
elapsed = time.time() - start

# Save final
pos = simulation.context.getState(getPositions=True).getPositions()
with open(final_pdb, 'w') as f:
    PDBFile.writeFile(simulation.topology, pos, f)

print(f'\nDone! {SIM_NS} ns in {elapsed/3600:.1f} hours')
print(f'Speed: {SIM_NS / (elapsed/86400):.1f} ns/day')
print(f'Results: {OUTPUT_DIR}')

In [ ]:
# Cell 6: Verify results
import csv
import statistics

print(f"{'='*60}")
print(f"VERIFICATION: {CANDIDATE_ID}")
print(f"{'='*60}\n")

# Check files
for label, path in [('Solvated', solvated_pdb), ('Trajectory', traj_file),
                     ('Energy', log_file), ('Final', final_pdb)]:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f'  [OK] {label}: {size:.1f} MB')
    else:
        print(f'  [MISSING] {label}')

# Parse energy log
if os.path.exists(log_file):
    temps = []
    times = []
    with open(log_file) as f:
        reader = csv.DictReader(f)
        for row in reader:
            for tc in ['Temperature (K)', 'temperature']:
                if tc in row:
                    try: temps.append(float(row[tc]))
                    except: pass
                    break
            for ttc in ['Time (ps)', 'time', '#"Time (ps)"']:
                if ttc in row:
                    try: times.append(float(row[ttc]))
                    except: pass
                    break

    if temps:
        print(f'\n  Temperature: {statistics.mean(temps):.1f} ± {statistics.stdev(temps):.1f} K')
    if times:
        ns_done = max(times) / 1000
        print(f'  Simulation time: {ns_done:.1f} / {SIM_NS} ns')
        if ns_done >= SIM_NS * 0.95:
            print(f'  [OK] Complete')
        else:
            print(f'  [WARNING] Incomplete — {ns_done:.1f}/{SIM_NS} ns')

print(f"\n{'='*60}")